# Reed-Solomon Decoder

In [90]:
%load_ext autoreload
%autoreload 1
%aimport classes.galois_field
%aimport classes.galois_poly
%aimport classes.rs_decoder

from classes.galois_field import *
from classes.galois_poly import *
from classes.rs_decoder import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [91]:
gf_extended = GaloisField(4, 0b10011)
code_roots  = [GFE1, 1, 2, 3, 4, 5]

decoder     = ReedSolomonDecoder(15, 9, gf_extended, code_roots)

#r   = 0x52D0A_F17CB_1EE64
#r   = 0x56D09_F171B_1E06F
r   = 0x52009F97C01E0C4
r   = decoder.do_unpack(r, 14)
eras_pos = [5, 12]
v   = decoder.decode(r, eras_pos)

print(r)
print(v)

Field Closed Succesfully!, 15 Non-Zero Elements
a13*x5 + a5*x4 + a3*x3 + a1*x2 + x + a2



UnboundLocalError: cannot access local variable 'Tx' where it is not associated with a value

## Annex B Equations Compared To Forney

In [ ]:
decoder = ReedSolomonDecoder(15, 9, gf_extended, code_roots)

# using shorten-code, zero message sent, only error patterns present
r   = 0x00D00_F0000_0E000
# r   = 0x00000_00000_00000
r   = decoder.do_unpack(r, 14)

print("Implicit Error Masks:")
print(r)

# get the locator poly
syn = decoder._get_syndromes(r)
Lx  = decoder._get_locator_poly(syn)

# get roots of reciprocal, error position
Lx          = Lx.get_reciprocal()
loc_root    = decoder._chien_search(Lx)

print()
print(Lx)

# forney to get the error mask
ek_forney   = decoder._get_error_mask(syn, Lx.get_reciprocal(), [gf_extended.inv(rk)[ID_POWER] for rk in loc_root])

# solve the system to get the error mask
# Si  == ek*(pk)^i
# 
# S0  == e0*1       + e1*1      + e2*1      => | S0 |   ==  | 1     , 1     , 1     || e0   |
# S1  == e0*p0      + e1*p1     + e2*p2     => | S1 |       | p0    , p1    , p2    || e1   |
# S2  == e0*p0^2    + e1*p1^2   + e2*p2^2   => | S2 |       | p0^2  , p1^2  , p2^2  || e2   |
r1  = GaloisPoly(gf_extended, [GFE1                                     , GFE1                                      , GFE1                                      , syn[0]])
r2  = GaloisPoly(gf_extended, [loc_root[0]                              , loc_root[1]                               , loc_root[2]                               , syn[1]])
r3  = GaloisPoly(gf_extended, [gf_extended.pow(loc_root[0],2)[ID_POWER] , gf_extended.pow(loc_root[1],2)[ID_POWER]  , gf_extended.pow(loc_root[2],2)[ID_POWER]  , syn[2]])

# reduce col 0
r2  = r2 + (r1 * (r2 @ 3))
r3  = r3 + (r1 * (r3 @ 3))
# reduce col 1
r1  = r2 + (r1 * (r2 @ 2))
r3  = (r3 * gf_extended.inv(r3 @ 2)[ID_POWER]) + (r2 * gf_extended.inv(r2 @ 2)[ID_POWER])
# reduce col 2
r1  = (r1 * gf_extended.inv(r1 @ 1)[ID_POWER]) + (r3 * gf_extended.inv(r3 @ 1)[ID_POWER])
r2  = r2 + r3
# normalize poly, get error mask
r1  = r1 * (gf_extended.inv(r1 @ 3)[ID_POWER])
r2  = r2 * (gf_extended.inv(r2 @ 2)[ID_POWER])
r3  = r3 * (gf_extended.inv(r3 @ 1)[ID_POWER])

print(r1)
print(r2)
print(r3)

ek_system   = [r1 @ 0, r2 @ 0, r3 @ 0]

# the complement of masks must be identical to the one by forney, and dentical to coeffs of r

ek_forney, ek_system, ek_forney == ek_system

Implicit Error Masks:
a13*x12 + a12*x9 + a11*x3

x3 + a13*x2 + a1*x + a9



TypeError: ReedSolomonDecoder._get_error_mask() missing 1 required positional argument: 'eras_pos'

In [ ]:
loc_root

[(15, 12), (10, 9), (8, 3)]